# Day 4 practical walkthrough: multilingual text analysis

**Research question.** Can we identify three kinds of climate-related political text—**policy**, **protest**, and **climate impact**—across languages, and how do our choices change the result?

We follow the lecture's sequence: (1) inspect the corpus, (2) choose where languages connect, (3) Route A: analyse originals separately and align outputs, (4) Route B: align inputs through translation, (5) Route C: use a shared multilingual model, and (6) evaluate by language and context.

The core notebook runs offline with the course requirements. Its 36 short English, German, and Spanish texts and English paraphrases were **written for teaching**; they are not parliamentary speeches, machine translations, or an empirical benchmark. A separate ParlLawSpeech section shows how to substitute authentic research data once a local extract is available. The supplied embedding file contains actual model outputs and is labelled as such. Optional model/API cells do not run automatically.

**Timing:** about 70–90 minutes for the core path with discussion; optional extensions can fill a longer workshop. Run cells in order. When you see an interpretation prompt, inspect the displayed rows before moving on.

## 0. Setup and data provenance

Every original text needs an ID, a language label, and a human label. We also retain the English teaching paraphrase as a *derived* text. Never overwrite the original with it. A real corpus would additionally store date, source, country, speaker and, where applicable, translation provenance.

In [ ]:
from pathlib import Path
import re, unicodedata
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.decomposition import NMF
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 115)

def find_data_dir():
    for candidate in [Path("materials/data"), Path("../data"), Path("data")]:
        if (candidate / "multilingual_climate_examples.csv").exists():
            return candidate
    raise FileNotFoundError("Start JupyterLab in the course root or materials/notebooks.")

DATA_DIR = find_data_dir()
texts = pd.read_csv(DATA_DIR / "multilingual_climate_examples.csv")
assert texts.id.is_unique and texts.notna().all().all()
assert set(texts.lang) == {"en", "de", "es"}
assert set(texts.gold) == {"policy", "protest", "impact"}
print("Documents:", len(texts), "| fields:", texts.columns.tolist())
display(texts.head(3))

In [ ]:
display(pd.crosstab(texts.lang, texts.gold, margins=True))
display(texts.groupby("lang").text.apply(lambda s: s.str.len().describe()[["min", "50%", "max"]]).unstack())

**Interpretation.** This sample is perfectly balanced by design: four examples per category per language. A real parliamentary corpus will not be. Its language mix, date range, institution, and speech length can all affect an apparent cross-language difference. Short headlines are useful for seeing errors, but they are easier than long speeches for some methods and harder for language detection.

### 0.1 Parallel, comparable, and translated are different

Our rows are *comparable* texts on the same broad topic. They are not parallel versions of the same speech. Each `translation_en` is a teaching paraphrase paired to its original. This is enough to compare route mechanics, but it cannot estimate a translation system's error rate. ParlLawSpeech has real parliamentary speeches and some translated fields; we revisit it at the end.

In [ ]:
display(texts.loc[texts.id.isin(["en02", "de02", "es02"]),
                  ["id", "lang", "text", "translation_en", "gold"]])

### 0.2 One label, a codebook, and the unit of analysis

The label refers to the **main event in the whole headline**. `policy` means a government measure or legislative debate; `protest` means collective public action; `impact` means a climate-related physical effect. A headline can mention several. We assign its main event, not every keyword. For long parliamentary speeches, sentence or paragraph units may be more appropriate; an entire speech can contain several topics and stances.

In [ ]:
display(texts.loc[texts.id.isin(["en10", "es10"]), ["id", "text", "translation_en", "gold"]])

Notice that both impact examples mention taxation. A simple tax dictionary can therefore score a false policy positive. Ask: would a multi-label codebook be more faithful for the full speech, and what would that change in the output table?

## 1. Inspect language and writing systems before selecting a route

Language ID is a measurement step. Source metadata may be more reliable than a detector for very short texts. Keep both, audit disagreements, and make an explicit decision for mixed-language documents.

### 1.1 Source labels, short texts, and code-switching

The following are *constructed diagnostic strings*. The point is to inspect where a one-language-per-row assumption breaks. A production language detector such as fastText can propose labels, but its confidence is not a calibrated probability of linguistic purity.

In [ ]:
diagnostics = pd.DataFrame([
    ("D1", "de", "Das Parlament diskutiert neue Emissionsregeln."),
    ("D2", "es", "El parlamento debate un impuesto al carbono."),
    ("D3", "mixed", "The committee diskutiert den carbon tax."),
    ("D4", "unknown", "EU climate deal"),
    ("D5", "zh", "议会讨论气候政策。"),
], columns=["id", "source_label", "text"])
display(diagnostics)

For D3, a document-level label such as `mixed` is more honest than forcing English or German. If the analysis requires local dictionaries, annotate language by sentence or span. Translation can help inspect a mixed span, but a translation of the whole document may hide the switch. D4 is too short to trust an automatic label without checking metadata.

In [ ]:
span_audit = pd.DataFrame([
    ("D3", "The committee", "en", "institutional subject"),
    ("D3", "diskutiert den", "de", "verb and determiner"),
    ("D3", "carbon tax", "en", "policy term"),
], columns=["document", "span", "language", "note"])
display(span_audit)

### 1.2 A deliberately small language-ID demonstration

This classifier learns character patterns from the 36 labelled teaching texts, then proposes languages for the diagnostic strings. It is **not** a credible production language detector: the training set is tiny, the German and Spanish cases resemble it, and mixed or unseen-script texts are outside its training distribution. The value of the exercise is to see why proposed labels and model confidence need manual review.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
lid_demo = make_pipeline(TfidfVectorizer(analyzer="char", ngram_range=(2,4)),
                         LogisticRegression(max_iter=1000))
lid_demo.fit(texts.text, texts.lang)
diagnostics["toy_prediction"] = lid_demo.predict(diagnostics.text)
diagnostics["top_model_score"] = lid_demo.predict_proba(diagnostics.text).max(axis=1).round(2)
display(diagnostics)

The `top_model_score` is the classifier's score under this toy training setup, **not** the probability that a document is single-language. D3 needs span review and D5 requires an out-of-scope or script check. In a real corpus, compare detector labels with source metadata and sample disagreements for inspection.

### 1.3 Unicode normalization and scripts

Unicode describes characters; a script is a writing system; language is another variable. German, Turkish and English use Latin script. Chinese text uses Han characters. Russian uses Cyrillic. Transliteration changes script, not meaning. Normalize to a documented form before matching strings, but retain the original text.

In [ ]:
surface = ["Café", "Cafe\u0301", "İstanbul", "Istanbul", "气候政策", "климат", "iklim"]
rows = []
for s in surface:
    rows.append({"text": s, "codepoints": " ".join(f"U+{ord(ch):04X}" for ch in s),
                 "NFC": unicodedata.normalize("NFC", s),
                 "casefold": s.casefold()})
display(pd.DataFrame(rows))
print("Two visually similar forms equal before NFC?", surface[0] == surface[1])
print("After NFC?", unicodedata.normalize("NFC", surface[0]) == unicodedata.normalize("NFC", surface[1]))

Casefolding is useful for matching but can have language-specific consequences. The Turkish dotted capital İ illustrates why a generic lowercase rule deserves inspection. Do not strip diacritics or transliterate by default: these transformations can merge distinct words.

### 1.4 Inspect tokens: German compounds, Turkish suffixes, Chinese boundaries

Whitespace segmentation is only a baseline. In German, `Emissionsregeln` contains a policy term without a space. Turkish attaches grammatical information to word stems. Chinese normally has no spaces between the words a researcher wants to count.

In [ ]:
examples = pd.DataFrame([
    ("en", "emission rules", "space-separated words"),
    ("de", "Emissionsregeln", "compound hides Regeln"),
    ("tr", "evlerimizden", "ev + ler + imiz + den: from our houses"),
    ("zh", "气候政策", "roughly climate policy; no space between the two concepts"),
    ("ru", "климатическая политика", "Cyrillic, with spaces"),
], columns=["lang", "text", "what_to_inspect"])
examples["split_on_spaces"] = examples.text.str.split()
display(examples)

**Decision.** For lexical methods, inspect actual tokens in each language before selecting stemming, lemmatization, compound splitting or stopwords. A multilingual transformer may use subword pieces, but those pieces are not automatically interpretable policy terms.

### 1.5 What a shared word TF–IDF representation sees

The next table is a deliberately tiny document-term matrix. The same policy idea has different surface forms. Chinese is an especially visible failure of whitespace-based word tokenization. `TfidfVectorizer`'s default token pattern is also a modeling choice; here we set a simple Unicode-aware token pattern explicitly.

In [ ]:
parallel_idea = pd.DataFrame([
    ("en", "Parliament approves emission rules"),
    ("de", "Parlament beschließt Emissionsregeln"),
    ("es", "El parlamento aprueba normas de emisiones"),
    ("zh", "议会批准排放规定"),
], columns=["lang", "text"])
word_vec = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", lowercase=True)
word_x = word_vec.fit_transform(parallel_idea.text)
display(pd.DataFrame(word_x.toarray().round(2), columns=word_vec.get_feature_names_out(), index=parallel_idea.lang))
display(pd.DataFrame(cosine_similarity(word_x).round(2), index=parallel_idea.lang, columns=parallel_idea.lang))

A zero or low cross-language cosine here measures surface overlap, not conceptual difference. Chinese appears as a long character sequence under this word rule. A word-level representation needs suitable segmentation; a character representation changes the feature definition, but still cannot translate unrelated expressions.

In [ ]:
char_vec = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), lowercase=True)
char_x = char_vec.fit_transform(parallel_idea.text)
display(pd.DataFrame(cosine_similarity(char_x).round(2), index=parallel_idea.lang, columns=parallel_idea.lang))

Character n-grams may capture cognates and spelling variants such as `emission` / `emisiones`, but there is no reason to expect them to align Chinese `排放` with English `emission`. This is a lexical baseline, not a shared semantic space.

## 2. Route A — analyse originals separately, align outputs

Route A preserves original-language text and routes each document to a local method. We align the *output categories*, not the input words. The common codebook lets us compare `policy`, `protest`, and `impact` across languages. Local methods may still have unequal error rates.

### 2.1 Language-specific dictionaries

These dictionaries are deliberately incomplete. They are a transparent starting point, not a validated measure. The classifier returns `uncoded` for no match and `review` for ties. We inspect those outcomes rather than hiding them.

In [ ]:
LEXICONS = {
 "en": {
  "policy": [r"tax\w*", r"rule\w*", r"subsid\w*", r"ban\w*", r"law\w*"],
  "protest": [r"activist\w*", r"march\w*", r"rally\w*", r"protest\w*", r"block\w*", r"demonstrat\w*"],
  "impact": [r"flood\w*", r"drought\w*", r"heatwave\w*", r"wildfire\w*", r"fire\w*"],
 },
 "de": {
  "policy": [r"steuer\w*", r"regel\w*", r"zusch\w*", r"verbot\w*", r"gesetz\w*"],
  "protest": [r"aktivist\w*", r"demonstr\w*", r"kundgebung\w*", r"protest\w*", r"block\w*"],
  "impact": [r"überschwemm\w*", r"dürre\w*", r"hitzewelle\w*", r"waldbrand\w*", r"waldbrände\w*"],
 },
 "es": {
  "policy": [r"impuesto\w*", r"norma\w*", r"subsidio\w*", r"prohibici\w*", r"ley\w*"],
  "protest": [r"activista\w*", r"marcha\w*", r"manifest\w*", r"protest\w*", r"bloque\w*"],
  "impact": [r"inundaci\w*", r"sequía\w*", r"ola\w*", r"incendio\w*"],
 }}
def tokens(text):
    return re.findall(r"[^\W\d_]+", unicodedata.normalize("NFC", str(text)).casefold(), flags=re.UNICODE)
def dictionary_scores(text, language, lexicons=LEXICONS):
    ts = tokens(text)
    return {label: sum(bool(re.fullmatch(p, t)) for p in patterns for t in ts)
            for label, patterns in lexicons[language].items()}
def dictionary_label(text, language, lexicons=LEXICONS):
    scores = dictionary_scores(text, language, lexicons)
    maximum = max(scores.values())
    if maximum == 0: return "uncoded"
    winners = [label for label, score in scores.items() if score == maximum]
    return winners[0] if len(winners) == 1 else "review"
texts["route_a_dict"] = [dictionary_label(row.text, row.lang) for row in texts.itertuples()]
display(texts[["id", "lang", "text", "gold", "route_a_dict"]])

### 2.2 Error audit and a local repair

Inspect the German compound and the Spanish heat expression. A broader pattern may raise coverage but can also produce false positives. Keep a before/after table and recheck all languages. Do not silently add terms after inspecting a test set if you intend the resulting accuracy to be an unbiased estimate.

In [ ]:
audit_a = texts.loc[texts.route_a_dict.ne(texts.gold), ["id", "lang", "text", "gold", "route_a_dict"]]
display(audit_a)
print("de02 tokens:", tokens(texts.loc[texts.id.eq("de02"), "text"].iloc[0]))
print("de02 scores:", dictionary_scores(texts.loc[texts.id.eq("de02"), "text"].iloc[0], "de"))

In [ ]:
from copy import deepcopy
REVISED_LEXICONS = deepcopy(LEXICONS)
REVISED_LEXICONS["de"]["policy"].append(r"emissionsregel\w*")
REVISED_LEXICONS["en"]["impact"].append(r"heat\w*")
texts["route_a_revised"] = [dictionary_label(row.text, row.lang, REVISED_LEXICONS)
                            for row in texts.itertuples()]
display(texts.loc[texts.route_a_dict.ne(texts.route_a_revised),
                  ["id", "lang", "text", "gold", "route_a_dict", "route_a_revised"]])

The changed rows are a *development-set illustration*. In a real project, freeze the revised dictionary and test it on unseen human-coded data. The English `heat` expansion also shows why broader lexical coverage needs a false-positive audit (`heat` can have non-climate meanings).

### 2.3 Entity linking across languages

A named-entity recognizer may find local mentions, but comparative analysis needs a **canonical ID**. The strings below are a constructed illustration; the same logic can be applied to parties, legislatures, institutions and people. Entity extraction and entity linking are distinct steps.

In [ ]:
mentions = pd.DataFrame([
 ("en", "European Commission", "EU_COMMISSION"),
 ("de", "Europäische Kommission", "EU_COMMISSION"),
 ("es", "Comisión Europea", "EU_COMMISSION"),
 ("en", "European Parliament", "EU_PARLIAMENT"),
 ("de", "Europäisches Parlament", "EU_PARLIAMENT"),
], columns=["lang", "surface", "entity_id"])
display(mentions)
print("Surface strings:", mentions.surface.nunique(), "linked entities:", mentions.entity_id.nunique())

### 2.4 Local supervised classifiers

The earlier class covered TF–IDF and supervised learning. Route A can fit one model per language using the same three output labels. Here, each model has only 12 documents, so leave-one-out predictions are a **mechanics demonstration**, not a stable performance estimate. The vectorizer is inside a pipeline so it is fit separately in each held-out fold. In a serious study, obtain substantially more source-language labels, keep related speeches together when splitting, and preserve an untouched test set.

In [ ]:
from sklearn.pipeline import make_pipeline
def local_loo_pipeline(frame):
    out = {}
    for lang, part in frame.groupby("lang"):
        pipe = make_pipeline(TfidfVectorizer(analyzer="char", ngram_range=(2,4)),
                             LogisticRegression(max_iter=1000))
        pred = cross_val_predict(pipe, part.text, part.gold, cv=LeaveOneOut())
        out.update(zip(part.id, pred))
    return pd.Series(out)
texts["route_a_local_ml_proper"] = texts.id.map(local_loo_pipeline(texts))
display(pd.crosstab(texts.lang, texts.route_a_local_ml_proper.eq(texts.gold),
                    rownames=["language"], colnames=["correct on held-out fold"]))

### 2.5 Local topic models and output alignment

Topic models discover word co-occurrence patterns; they do not inherit the shared `policy/protest/impact` codebook. The next cell fits one small NMF model per language. Topic numbers are arbitrary across fits. Read top words **and representative documents** before giving any topic a shared label. With 12 short texts per language, expect unstable topics.

In [ ]:
local_topics = {}
for lang, part in texts.groupby("lang"):
    vec = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", min_df=1)
    X = vec.fit_transform(part.text)
    nmf = NMF(n_components=3, random_state=7, init="nndsvda", max_iter=800)
    weights = nmf.fit_transform(X)
    terms = vec.get_feature_names_out()
    local_topics[lang] = (part, weights, nmf, terms)
    print("\n", lang)
    for k, component in enumerate(nmf.components_):
        top = terms[np.argsort(component)[-6:][::-1]]
        exemplar = part.iloc[weights[:, k].argmax()]
        print(f"topic {k}: {', '.join(top)} | example {exemplar.id}: {exemplar.text}")

**Interpretation.** Do any three topics map cleanly to our codebook categories? If not, do not force them. An output-alignment protocol could have bilingual reviewers read top terms and representative speeches, assign a shared concept where defensible, and mark ambiguous topics. Local topics can be useful even when no one-to-one mapping exists.

### 2.6 Why an English sentiment lexicon is not a multilingual default

Consider the same policy evaluation in three languages. A word list containing only `good` and `bad` gives artificial zeros for German and Spanish. Translation or local dictionaries can help, but sentiment in political speech depends on target and context: approving a restriction and approving pollution are different positions.

In [ ]:
sentiment_demo = pd.DataFrame([
 ("en", "The new rule is good"),
 ("de", "Die neue Regel ist gut"),
 ("es", "La nueva norma es buena"),
], columns=["lang", "text"])
english_lexicon = {"good": 1, "bad": -1}
sentiment_demo["english_only_score"] = sentiment_demo.text.apply(
    lambda s: sum(english_lexicon.get(t, 0) for t in tokens(s)))
display(sentiment_demo)

### 2.7 Cross-language sentiment: the three-route decision

Sentiment is a separate construct from the headline's topic. The small table below makes the same positive/negative evaluation in English, German and Spanish. Route A uses local words; Route B uses English paraphrases. A shared multilingual sentiment model would be Route C, but it requires task-specific validation. Even a correct **valence** label does not tell us whether the speaker supports a climate policy: the target of evaluation matters.

In [ ]:
sentiment_pairs = pd.DataFrame([
 ("en", "The climate rule is good", "The climate rule is good", "positive"),
 ("de", "Die Klimaregel ist gut", "The climate rule is good", "positive"),
 ("es", "La norma climática es buena", "The climate rule is good", "positive"),
 ("en", "The climate rule is bad", "The climate rule is bad", "negative"),
 ("de", "Die Klimaregel ist schlecht", "The climate rule is bad", "negative"),
 ("es", "La norma climática es mala", "The climate rule is bad", "negative"),
], columns=["lang", "original", "English_paraphrase", "human_valence"])
local_words = {"en": {"good": 1, "bad": -1},
               "de": {"gut": 1, "schlecht": -1},
               "es": {"buena": 1, "mala": -1}}
def valence_score(s, lex): return sum(lex.get(t, 0) for t in tokens(s))
sentiment_pairs["A_local"] = [valence_score(r.original, local_words[r.lang])
                               for r in sentiment_pairs.itertuples()]
sentiment_pairs["B_English_paraphrase"] = [valence_score(s, local_words["en"])
                                          for s in sentiment_pairs.English_paraphrase]
sentiment_pairs["English_only_on_original"] = [valence_score(s, local_words["en"])
                                               for s in sentiment_pairs.original]
display(sentiment_pairs)

The six paraphrases are constructed and perfectly literal; this table shows route mechanics, not translation quality. For Route C, a multilingual sentiment classifier can read the originals directly. Evaluate it against human valence labels separately by language and by target. An optional model call appears below; no download is triggered in the core path.

In [ ]:
RUN_SENTIMENT_MODEL = False
if RUN_SENTIMENT_MODEL:
    from transformers import pipeline
    sentiment_model = pipeline(
        "text-classification",
        model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
        truncation=True,
    )
    sentiment_pairs["C_model_output"] = [r["label"] for r in
        sentiment_model(sentiment_pairs.original.tolist())]
    display(sentiment_pairs[["lang", "original", "human_valence", "C_model_output"]])

## 3. Route B — translate inputs, then analyse in one target language

Route B creates a common English input before applying a method. Our `translation_en` column is a human-written teaching paraphrase, **not machine translation output**. It lets us demonstrate what a translation-first pipeline would do without a service, and it makes label-critical changes inspectable.

### 3.1 Word translation versus document translation

Translating only search terms can expand a dictionary while the analysis stays on original text (Route A). Translating each full document creates English input for a target-language method (Route B). In a production pipeline, save source text, translated text, provider/model/version, date, and any manual corrections.

In [ ]:
translation_units = pd.DataFrame([
 ("term", "Emissionsregeln", "emissions rules", "Route A dictionary expansion"),
 ("sentence", "Das Parlament beschließt neue Emissionsregeln",
  "Lawmakers adopt new emissions standards", "Route B input alignment"),
], columns=["unit", "German original", "English rendering", "use"])
display(translation_units)

### 3.2 Apply one English dictionary to every paraphrase

We deliberately reuse the *baseline* English dictionary rather than adding terms after seeing translated errors. This makes differences between Routes A and B visible. Source-language human labels remain the reference.

In [ ]:
texts["route_b_dict"] = texts.translation_en.apply(lambda s: dictionary_label(s, "en"))
display(texts[["id", "lang", "text", "translation_en", "gold", "route_b_dict"]])

### 3.3 Inspect paired label changes

A translated phrase can make a previously hidden concept match. It can also replace a matching phrase with a synonym the English method misses. Compare the original and derived texts row by row before interpreting any summary statistic.

In [ ]:
changed = texts.loc[texts.route_a_dict.ne(texts.route_b_dict),
                    ["id", "lang", "text", "translation_en", "gold", "route_a_dict", "route_b_dict"]]
display(changed)

Look for `de02`: German `Emissionsregeln` is missed by the baseline local dictionary, while the English paraphrase includes `standards`, which the English dictionary may also miss. For `de11`, the German heatwave expression matches locally, but `Extreme heat` falls outside the baseline English impact patterns. The crucial issue is whether *the evidence for the research label* survives translation. General translation fluency scores do not answer that question.

### 3.4 Translate-test versus translate-train

- **Translate-test:** train an English model on English labelled texts, translate non-English evaluation texts to English, and apply the English model.
- **Translate-train:** translate labelled training examples into the target language, train a target-language model, then apply it to original target-language text.

The toy cell below demonstrates translate-test only. It trains on 12 English texts and applies the trained pipeline to 24 English paraphrases. There is no independent test set and the teaching paraphrases resemble English training texts; treat predictions as a workflow demonstration.

In [ ]:
english_train = texts.loc[texts.lang.eq("en")]
other_test = texts.loc[texts.lang.ne("en")]
english_pipe = make_pipeline(TfidfVectorizer(analyzer="char", ngram_range=(2, 5)),
                             LogisticRegression(max_iter=1000))
english_pipe.fit(english_train.text, english_train.gold)
texts["route_b_ml"] = english_pipe.predict(texts.translation_en)
display(texts.loc[texts.lang.ne("en"),
                  ["id", "lang", "translation_en", "gold", "route_b_ml"]].head(12))

### 3.5 One pooled target-language topic model

Translation also allows a single English topic model. It changes the input corpus, not the definition of a topic. Inspect original texts alongside English representative documents: the topic may reflect translation choices or source contexts rather than a stable theme.

In [ ]:
vec_b = TfidfVectorizer(stop_words="english", min_df=2)
X_b = vec_b.fit_transform(texts.translation_en)
nmf_b = NMF(n_components=3, random_state=7, init="nndsvda", max_iter=800)
W_b = nmf_b.fit_transform(X_b)
terms_b = vec_b.get_feature_names_out()
for k, row in enumerate(nmf_b.components_):
    top = terms_b[np.argsort(row)[-7:][::-1]]
    exemplar = texts.iloc[W_b[:, k].argmax()]
    print(f"Topic {k}: {', '.join(top)} | {exemplar.id}: {exemplar.text}")

### 3.6 A translation audit table

These are **constructed diagnostic cases**, not observed errors from a translation service. Record both a fluent translation and whether the feature needed for coding survives. Names of institutions, negation, modality, and local policy terms deserve special attention.

In [ ]:
translation_audit = pd.DataFrame([
 ("negation", "Der Ausschuss lehnt die Steuer nicht ab.",
  "The committee does not reject the tax.", "Does 'not' survive?"),
 ("institution", "Der Bundesrat stimmt zu.",
  "The Federal Council agrees.", "Which Bundesrat / entity ID?"),
 ("compound", "Die Emissionsregeln gelten ab Januar.",
  "The emissions standards apply from January.", "Does policy meaning survive?"),
 ("local context", "El cabildo aprueba la medida.",
  "The council approves the measure.", "Which local institution is meant?"),
], columns=["risk", "original", "illustrative English", "label-critical check"])
display(translation_audit)

## 4. Route C — shared multilingual representations and models

A multilingual sentence encoder maps original-language texts to vectors in a shared 384-dimensional space. Such spaces are learned through multilingual pretraining and, for sentence alignment models, cross-language matching objectives. Similarity can support retrieval and clustering; it is **not** itself a policy label or a stance judgement.

The bundled `.npz` contains embeddings computed from the 36 original texts using `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`. The model was run once and the vectors were saved so the core practical works offline. This asset is genuine model output, unlike the teaching paraphrases.

In [ ]:
with np.load(DATA_DIR / "multilingual_climate_embeddings.npz") as asset:
    embedding_ids = asset["ids"].astype(str)
    vectors = asset["embeddings"]
assert embedding_ids.tolist() == texts.id.tolist()
assert vectors.shape == (len(texts), 384)
print("Embedding matrix:", vectors.shape)
print("First vector coordinates:", np.round(vectors[0, :8], 3))
print("Vector length:", np.linalg.norm(vectors[0]).round(3))

### 4.1 Cross-language retrieval on original text

Choose one English query. Rank **German and Spanish originals only** by cosine similarity. Then read the texts. Retrieval can surface translations or related policy themes without word overlap, but a high score does not prove that two statements make the same claim.

In [ ]:
query_id = "en02"
qi = texts.index[texts.id.eq(query_id)][0]
scores = cosine_similarity(vectors[qi:qi+1], vectors).ravel()
retrieval = texts.loc[texts.lang.ne("en"), ["id", "lang", "text", "gold"]].copy()
retrieval["cosine"] = scores[retrieval.index]
print("Query:", texts.loc[qi, "text"])
display(retrieval.sort_values("cosine", ascending=False).head(8))

### 4.2 Similarity is not a category or stance label

Inspect nearest neighbours for a protest text. Some neighbours may share the event type; others may share vocabulary or political setting. If you seek stance toward a carbon tax, define stance labels and evaluate those labels separately. Embeddings answer a similarity question, which may or may not match the substantive question.

In [ ]:
def nearest_other_languages(document_id, n=5):
    i = texts.index[texts.id.eq(document_id)][0]
    sims = cosine_similarity(vectors[i:i+1], vectors).ravel()
    out = texts.loc[texts.lang.ne(texts.loc[i, "lang"]), ["id", "lang", "text", "gold"]].copy()
    out["cosine"] = sims[out.index]
    return out.sort_values("cosine", ascending=False).head(n)
print("Query:", texts.loc[texts.id.eq("en07"), "text"].iloc[0])
display(nearest_other_languages("en07"))

### 4.3 Zero-shot cross-language classification with prototypes

Here, “zero-shot” means **no German or Spanish labelled training examples**. We average English vectors within each category to form three prototypes, then assign each German/Spanish text to the closest prototype. This is a transparent baseline, not a fine-tuned transformer or NLI classifier. Measure how it performs by language; cross-language similarity does not guarantee equal label accuracy.

In [ ]:
categories = ["policy", "protest", "impact"]
en_mask = texts.lang.eq("en").to_numpy()
prototypes = np.vstack([vectors[en_mask & texts.gold.eq(label).to_numpy()].mean(axis=0)
                        for label in categories])
prototype_scores = cosine_similarity(vectors, prototypes)
texts["route_c_proto"] = np.array(categories)[prototype_scores.argmax(axis=1)]
for j, label in enumerate(categories): texts[f"similarity_{label}"] = prototype_scores[:, j]
display(texts.loc[texts.lang.ne("en"),
                  ["id", "lang", "gold", "route_c_proto"] + [f"similarity_{x}" for x in categories]].head(12))

### 4.4 Multilingual labels and fine-tuning

The prototype test above gives English labels only. In a real study, labelled examples from each target language can improve and reveal uneven performance. A multilingual transformer can be fine-tuned end-to-end for the shared codebook, but that requires a larger labelled corpus and train/dev/test splits that prevent duplicate speeches, translated versions, or speakers leaking across sets. We do not fine-tune on this 36-row sample because the exercise would teach a misleading result.

### 4.5 Shared multilingual topics and clusters

A multilingual topic workflow embeds originals, groups vectors, and then interprets groups by reading representative documents in *each* language. BERTopic packages a similar broad pipeline with dimensionality reduction, clustering and topic descriptions. Here we use KMeans as a small inspectable example. Cluster IDs are not substantive topic labels, and the choice of three clusters is imposed by us.

In [ ]:
clusterer = KMeans(n_clusters=3, random_state=7, n_init=10)
texts["shared_cluster"] = clusterer.fit_predict(vectors)
display(pd.crosstab(texts.shared_cluster, texts.lang))
for cluster in sorted(texts.shared_cluster.unique()):
    part = texts.loc[texts.shared_cluster.eq(cluster)]
    print(f"\nCluster {cluster} ({len(part)} texts)")
    display(part.groupby("lang", group_keys=False).head(2)[["id", "lang", "text", "gold"]])

A polylingual topic model is a different design: it uses **linked versions of the same document** across languages. Our 36 originals are comparable but not parallel documents, so we cannot honestly demonstrate a polylingual model with this table. If a corpus has aligned language versions of the same EP speech or law, the document link becomes a modeling input. Keep that distinction clear when choosing data.

### 4.6 What linked documents add to a polylingual model

The table below is constructed solely to show **input alignment** for a polylingual topic model. Each row is one document represented in three languages, so the model can use the document link. Our main headline corpus has no such link between languages. The data structure is the key difference from merely fitting topics on a pool of unrelated multilingual texts.

In [ ]:
linked_example = pd.DataFrame([
 ("doc_1", "Parliament approves emission rules",
  "Das Parlament beschließt Emissionsregeln",
  "El parlamento aprueba normas de emisiones"),
 ("doc_2", "Protesters block the city centre",
  "Demonstrierende blockieren die Innenstadt",
  "Manifestantes bloquean el centro urbano"),
], columns=["document_id", "en", "de", "es"])
display(linked_example.set_index("document_id"))
print("Linked document IDs:", linked_example.document_id.tolist())

### 4.7 Optional zero-shot classification with multilingual NLI

An NLI model evaluates whether each label description follows from the text. It can classify a text without task-specific training examples, but label wording, hypothesis template and model language coverage affect results. This is a **different classifier** from our embedding prototypes. The following cell is off by default because it needs `transformers`, `torch`, `sentencepiece`, and a local/downloadable multilingual NLI model. If enabled, test it against the same human labels by language.

In [ ]:
RUN_NLI = False
if RUN_NLI:
    from transformers import pipeline
    nli = pipeline("zero-shot-classification",
                   model="joeddav/xlm-roberta-large-xnli")
    candidate_labels = ["a government climate policy",
                        "a public climate protest",
                        "a physical climate impact"]
    example = texts.loc[texts.id.eq("de02"), "text"].iloc[0]
    nli_result = nli(example, candidate_labels=candidate_labels,
                     hypothesis_template="This text concerns {}.")
    display(pd.DataFrame({"label": nli_result["labels"],
                          "score": nli_result["scores"]}))

### 4.8 LLM coding of original-language text

An LLM can apply the shared codebook to source-language text. It belongs under the same three routes: it may read originals in a shared multilingual setup (C), read translations (B), or support local-language workflows (A). Prompt language, input language, category definitions, and output language are separate choices. The following code **constructs prompts only**; it does not send text to an API.

In [ ]:
CODEBOOK = """Assign the main event in the headline to exactly one label:
policy = government measure or legislative debate;
protest = collective public action;
impact = physical climate-related effect.
If evidence is insufficient, return uncertain. Output only one label."""
def make_prompt(row, instructions_language="en", input_column="text"):
    if instructions_language != "en":
        raise ValueError("This teaching template supplies English instructions only.")
    return f"{CODEBOOK}\nLanguage: {row['lang']}\nText: {row[input_column]}\nLabel:"
print(make_prompt(texts.loc[texts.id.eq("de10")].iloc[0]))

For a proper prompt-language experiment, translate the **instructions carefully** and keep the codebook, examples, model version, decoding settings and test set fixed. The old blanket claim “English prompts are better” is too broad; evidence depends on model, task and language. Do not assume that fluency in a language means valid annotation of the local political context.

**Hands-on annotation.** Copy the following three prompts into the same permitted model, keeping model version and settings fixed. Paste only the returned category strings into `manual_llm_labels` below. Compare the result to the source-language human labels. This separates actual observed model responses from examples written by us.

In [ ]:
llm_sample_ids = ["en10", "de10", "es10"]
for document_id in llm_sample_ids:
    row = texts.loc[texts.id.eq(document_id)].iloc[0]
    print("\n---", document_id, "---\n", make_prompt(row))

In [ ]:
manual_llm_labels = {"en10": None, "de10": None, "es10": None}
if all(label in {"policy", "protest", "impact", "uncertain"}
       for label in manual_llm_labels.values()):
    llm_observed = texts.loc[texts.id.isin(manual_llm_labels),
                             ["id", "lang", "text", "gold"]].copy()
    llm_observed["model_label"] = llm_observed.id.map(manual_llm_labels)
    llm_observed["correct"] = llm_observed.gold.eq(llm_observed.model_label)
    display(llm_observed)
else:
    print("Add the three observed model labels above to evaluate this optional activity.")

### 4.9 An inspectable LLM evaluation protocol

The next table is a *blank recording sheet*, not fabricated model output. If you have a model and permission to use it, run each prompt condition on the same held-out source-language texts and fill `prediction` with the actual returned label. Keep invalid or noncompliant responses visible rather than dropping them.

In [ ]:
llm_protocol = pd.DataFrame([
 ("original text / English instructions", "text"),
 ("English translation / English instructions", "translation_en"),
 ("original text / local instructions", "text"),
], columns=["condition", "input_field"])
llm_protocol["model_version"] = "record exact model/version"
llm_protocol["prediction"] = "to be collected"
display(llm_protocol)

**Research connection.** Published work finds that multilingual annotation results vary by task and language: genre classification, sentiment, and psychological coding do not exhibit one uniform performance pattern. Unequal language representation in model training and geographical skew in web data make per-language and per-context audits necessary. See the readings at the end.

## 5. Compare methods against source-language human labels

The same human-coded `gold` labels are the reference for all routes. Coverage, accuracy and error type answer different questions. This sample's small equal cells are valuable for inspection but cannot support population claims. Only the dictionary, translated-dictionary and prototype outputs are directly compared below; the local classifier is shown separately because its training/evaluation setup differs.

In [ ]:
method_cols = {
    "A: local dictionaries": "route_a_dict",
    "B: English paraphrase + dictionary": "route_b_dict",
    "C: multilingual embedding prototypes": "route_c_proto",
}
long = pd.concat([texts[["id", "lang", "gold", col]].rename(columns={col:"prediction"})
                  .assign(method=name) for name, col in method_cols.items()], ignore_index=True)
long["correct"] = long.prediction.eq(long.gold)
long["coded"] = long.prediction.isin(categories)
summary = long.groupby(["method", "lang"], sort=False).agg(
    n=("id", "size"), coverage=("coded", "mean"), accuracy=("correct", "mean"))
display(summary.round(2))

The prototype classifier scores **100% on this constructed, closely matched sample**. This is not evidence of equivalent performance across languages or on parliamentary speeches. It is exactly why we need unseen, independently sampled, source-language human labels before choosing a method.

### 5.1 Error matrices, not only pooled accuracy

A category-specific miss may matter more than a small change in overall accuracy. Check whether the method is systematically weaker for one language or one category. In a larger study, report confidence intervals and enough examples for each stratum.

In [ ]:
for method, block in long.groupby("method", sort=False):
    print("\n", method)
    display(pd.crosstab([block.lang, block.gold], block.prediction,
                        dropna=False).reindex(columns=categories+["uncoded", "review"], fill_value=0))

### 5.2 Read disagreements in the original language

The disagreement table links outputs back to the source text. Inspect the original and, for Route B, the paraphrase. Some disagreements are model errors; others reveal an ambiguous codebook or an input change. A disagreement is a good sampling device for human review, not proof that the majority method is correct.

In [ ]:
outputs = texts[["id", "lang", "text", "translation_en", "gold"] + list(method_cols.values())].copy()
outputs["distinct_outputs"] = outputs[list(method_cols.values())].nunique(axis=1)
display(outputs.loc[outputs.distinct_outputs.gt(1)].sort_values(["lang", "id"]))

### 5.3 Thresholds and an uncertain category

A forced prototype label may hide weak evidence. We can abstain when the top two similarities are too close. The `0.03` margin below is **illustrative, chosen without tuning**. A defensible threshold must be selected on a development set and tested on a separate held-out set, possibly by language if that choice is justified and documented.

In [ ]:
ordered = np.sort(prototype_scores, axis=1)
texts["prototype_margin"] = ordered[:, -1] - ordered[:, -2]
texts["route_c_with_abstention"] = np.where(
    texts.prototype_margin < 0.03, "uncertain", texts.route_c_proto)
display(texts.groupby("lang").agg(
    uncertain=("route_c_with_abstention", lambda s: s.eq("uncertain").sum()),
    mean_margin=("prototype_margin", "mean")).round(3))

### 5.4 Country, language and context can be confounded

The toy corpus has language but no country. In real data, a German-language subset might be mostly German parliamentary speeches and a Spanish-language subset mostly Spanish parliamentary speeches. A difference could reflect country, institution, time, topic mix, speaking style, or method error. Record enough metadata and design overlapping comparisons before interpreting it as a language effect.

In [ ]:
design_matrix = pd.DataFrame([
 ("Germany", "German", "Bundestag", "2018–2021"),
 ("Spain", "Spanish", "Congress", "2018–2021"),
 ("EU", "German", "European Parliament", "2018–2021"),
 ("EU", "Spanish", "European Parliament", "2018–2021"),
], columns=["country_or_body", "language", "institution", "period"])
display(design_matrix)

This **illustrative design matrix contains no observed corpus counts**. The EP rows show how an institution with multiple languages can help separate language from national legislature, but member state, topic and speaker still matter.

## 6. Put authentic data into the same pipeline: ParlLawSpeech

[ParlLawSpeech](https://parllawspeech.org/data) contains speeches, bills and laws for seven European countries and the European Parliament, with `procedure_ID` links and rich metadata. It is a much better *research corpus* than our 36 teaching headlines. It is **not** a ready-made three-category benchmark: its speech texts do not come with our `policy/protest/impact` human labels, and many speeches cover multiple issues. The release is around 4 GB in RDS format, so downloading it during class would be fragile. Prepare a small, licensed extract in advance.

The European Parliament file includes multilingual material and some English translations, making it particularly useful for Route B and language-ID audits. National parliamentary files let students compare other languages and metadata, but country and language often coincide. The code below is a schema adapter; it runs only if you place a pre-extracted CSV/Parquet at `materials/data/parllawspeech_sample.csv` or `.parquet`. It never silently substitutes constructed text for real PLS data.

In [ ]:
pls_csv = DATA_DIR / "parllawspeech_sample.csv"
pls_parquet = DATA_DIR / "parllawspeech_sample.parquet"
if pls_parquet.exists():
    pls = pd.read_parquet(pls_parquet)
elif pls_csv.exists():
    pls = pd.read_csv(pls_csv)
else:
    pls = None
    print("No ParlLawSpeech extract bundled. The core walkthrough is complete above.")
if pls is not None:
    print("PLS extract rows:", len(pls))
    print("Available columns:", pls.columns.tolist())
    display(pls.head(3))

### 6.1 Prepare a small PLS extract before class

Download the official [GESIS release](https://doi.org/10.7802/2824), read the relevant RDS file with R or `pyreadr`, and export a manageable sample. Preserve at least `speechnumber`, `text`, `date`, `party`, `speaker`, `agenda`, `procedure_ID`, the source legislature, language metadata if present, and the dataset's English translation where present. Document the exact release, selection rule and any language-detection step. Avoid taking only speeches that contain known climate words if the goal is to estimate prevalence: that would build the answer into the sample.

For this class, a workable sample could stratify by legislature, year and speech length, then deliberately oversample a small set of climate-related speeches for a *method demonstration*. Mark the oversampling flag and never report its category proportions as population prevalence. Human-code a separate stratified test set before comparing methods.

In [ ]:
if pls is not None:
    required = {"text", "speechnumber"}
    missing = required - set(pls.columns)
    if missing:
        print("Required columns missing:", missing)
    else:
        pls = pls.dropna(subset=["text"]).copy()
        pls["n_chars"] = pls.text.astype(str).str.len()
        display(pls.n_chars.describe().to_frame("speech characters"))
        if "date" in pls.columns:
            display(pls.assign(year=pd.to_datetime(pls.date, errors="coerce").dt.year)
                      .groupby("year", dropna=False).size().tail(10).to_frame("speeches"))
        if "lang" in pls.columns:
            display(pls.lang.value_counts(dropna=False).to_frame("speeches"))

### 6.2 Apply the methods to an extract only after checking the unit

A full parliamentary speech may contain a bill discussion, a protest reference and an account of climate impacts. The one-label headline codebook is not directly transferable. First choose a unit (speech, paragraph, sentence), write a new coding guide, and create a human-coded sample. The `procedure_ID` link can connect speeches with bills and enacted laws for contextual reading, but it is not itself a topic or stance label.

## 7. Optional: regenerate the bundled embeddings locally

Run this cell only when the sentence-transformers package and cached/downloadable model are available. It computes embeddings from **original-language** texts and compares them with the bundled asset. Do not run it during class if downloads are unreliable. It does not require an API key. Model provenance: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`; the course asset was computed with normalized embeddings and saved as float32.

In [ ]:
RUN_MODEL = False
if RUN_MODEL:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    regenerated = model.encode(texts.text.tolist(), normalize_embeddings=True,
                               show_progress_bar=False)
    print("Shape:", regenerated.shape)
    print("Largest absolute coordinate difference:",
          np.max(np.abs(regenerated.astype("float32") - vectors)))

## 8. Research design handoff

Write five concrete lines for your own project:

1. **Corpus/unit:** which documents and what unit will receive one label?
2. **Construct:** what exactly is the shared codebook category, and who can adjudicate local meanings?
3. **Alignment:** Route A outputs, Route B inputs, Route C shared model, or a deliberate combination?
4. **Human evidence:** which source-language items will be coded independently of method development?
5. **Validation:** which per-language, per-country and per-context errors would threaten the claim?

The model comparison in this notebook is a rehearsal of that design, not a result about real parliamentary discourse.

## References and data

- Schwalbach, Hetzer, Proksch, Rauh & Sebők, [ParlLawSpeech data release](https://doi.org/10.7802/2824), version 1.0.0; [data and schema](https://parllawspeech.org/data).
- Feng et al., [Language-agnostic BERT Sentence Embedding](https://aclanthology.org/2022.acl-long.62/): one approach to sentence alignment.
- Mimno et al., [Polylingual Topic Models](https://aclanthology.org/D09-1092/): linked multilingual documents.
- Kuzman et al., [multilingual genre annotation](https://arxiv.org/abs/2303.03953).
- Bang et al., [multilingual sentiment evaluation](https://aclanthology.org/2023.ijcnlp-main.45/).
- Rathje et al., [psychological coding across languages](https://www.pnas.org/doi/10.1073/pnas.2308950121).
- Bareiß et al., [prompt language and emotion classification](https://arxiv.org/abs/2402.03223).

**Teaching data:** `multilingual_climate_examples.csv` (36 constructed original headlines and English paraphrases), `multilingual_climate_embeddings.npz` (model-generated vectors for those originals). Neither is a natural sample of political speech.